# Language Modeling with Transformers
In the previous lab, we modeled language using N-grams, where the probability of a word depends only on a limited context of previous tokens. 
However, this approach cannot capture dependencies beyond the N-gram size, and increasing the order N is not possible after a certain point due to data sparsity.

How can we build models that can better model the context of sequences?

In this lab, we move to Transformer-based language models, where the model learns which parts of the context are important for predicting the next word.

### Objectives
- How can language be modeled using Transformers?
- How can we train and evaluate a neural language model?
- How can we efficiently fine-tune large models with pre-trained weights?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/massimo-rizzoli/NLU-2026-Labs/blob/main/labs/04_LM_with_transformers.ipynb)

### Recommended Reading

- Dan Jurafsky and James H. Martin's __Speech and Language Processing__ ([3rd ed. draft](https://web.stanford.edu/~jurafsky/slp3/))

- [Chapter 3: N-gram Language Models](https://web.stanford.edu/~jurafsky/slp3/3.pdf) (Language Modeling, Perplexity)
- [Chapter 8: Transformers](https://web.stanford.edu/~jurafsky/slp3/8.pdf) 

# 1 Language Models with Neural Networks

While we have already seen a language model based on N-grams, in this lab we are going to develop a language model using a neural architecture. 

## 1.1 Task definition

To model the probability distribution over a sequence, we are going to use the Chain Rule as we have seen in LAB 3:

$$P(w_{1}^{n}) = P(w_1) P(w_2|w_1) P(w_3|w_1^2) ... P(w_n|w_{1}^{n-1}) = \prod_{i=1}^{n}{P(w_i|w_{1}^{i-1})}$$

However, at that time we used ngram to truncate the previous context ($N-1$), to compute meaningful probabilities. While using neural models, we will let the model decide by itself how to manage the previous context and thus tokens which are relevant for the prediction.


## 1.2 Transformers

One of the most suitable neural architectures for the Language Model task is the Transformer. In this, we are going to consider Decoder-only models such as GPT2.

The architecture is composed of a series of transformer blocks followed by an output layer. 
The transformer blocks consist of linear layers that can process the tokens of the input sequence in parallel, and the masked self-attention mechanism that allows to connect the representations of the tokens.
The output layer is a linear+softmax that outputs the probability over the vocabulary. Indeed, the size of the output vector is equal to the size of the vocabulary, i.e. the model cannot predict tokens that are not present in the vocabulary. <br>


> LM task with Transformers can be tackled as a sequence labelling task (each input token has an output label) in which the input sequence is $ input = \{w_1, w_2, \cdots, w_{n-1}\}$ and the output is $ output = \{w_2, w_3, \cdots, w_{n}\}$



***Example***:
 > For the input sentence ***"I go to Miami"***, the input sequence of the model is ***"I go to"*** and the target/output sequence is ***"go to Miami"***.


<p align="center">
    <img src="https://i.postimg.cc/Px1qXDbQ/Transformer-Example-2.png" alt="drawing" width="300"/>
</p>
In the image below you can see a working example of language model based on a transformer. 
<p align="center">
    <img src="https://i.postimg.cc/FzhBftSw/Transformer-Example.png" alt="drawing" width="300"/>
</p>


# 2 Model architecture

We provide a simplified implementation of GPT2 inspired by [minGPT](https://github.com/karpathy/minGPT) and [GPT2LMHeadModel from Huggingface](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel).

Here we define the architecture of our model using PyTorch. In the `__init__` method, we define the class of our model and we instantiate all the layers that we are going to use. In the `forward` method we define the interactions among the instantiated layers, in other words, we design the architecture of the model.   

In [17]:
# If you are using Colab, run this line, then restart the runtime
#%pip install transformers==4.38.0

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 2.1 Multi-Head Attention

Attention is computed for each head. Each head has its own learnable parameters.
<p align="left">
    <img src="https://i.postimg.cc/G2Y0NBZT/single-head.png" alt="drawing" width="800"/>
</p>

The outputs of the heads are concatenated and passed through a learnable projection layer.

<p align="left">
    <img src="https://i.postimg.cc/vHyyJzYK/multi-head.png" alt="drawing" width="800"/>
</p>


GPT2 is a decoder-only model! We use **masked** self-attention to allow the model only to condition on the previous tokens:
<p align="left">
    <img src="https://i.postimg.cc/xdPn5pbq/masked-attention-full.png" alt="drawing" width="800"/>
</p>

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.h_dim = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        # batch size B, sequence length L, d_model
        B, L, d_model = x.size() 

        q = self.w_q(x) # (B, L, d_model)
        k = self.w_k(x) # (B, L, d_model)
        v = self.w_v(x) # (B, L, d_model)

        # reshape to (B, n_heads, L, h_dim)
        q = q.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)

        # dot-product between each head's q and k matrices
        # (B, n_heads, L, h_dim) * (B, n_heads, h_dim, L) -> (B, n_heads, L, L), i.e., similarities for each head
        similarity = q @ k.transpose(-2, -1) # transpose(-2,-1) transposes the k matrix for each head

        # normalize
        similarity = similarity * (1 / torch.sqrt(torch.tensor(self.h_dim)))

        # mask
        similarity = similarity.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(similarity, dim=-1)

        y = attn @ v  # (B, n_heads, L, L) * (B, n_heads, L, h_dim) -> (B, n_heads, L, h_dim)
        y = y.transpose(1, 2) # (B, L, n_heads, h_dim)
        # concatenate outputs of each head (contiguous is necessary to use view())
        y = y.contiguous().view(B, L, d_model) # (B, L, d_model)
        y = self.out_proj(y)

        return y

## 2.2 Feed Forward

The Feed Forward of GPT2 is similar to that of the original transformer architecture:
<p align="left">
    <img src="https://i.postimg.cc/FHM4h50r/feed-forward.png" alt="drawing" width="800"/>
</p>

But GPT2 uses **GELU** instead of ReLU as the activation function.

In [20]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model),
        )

    def forward(self, x):
        # applied over each token independently
        return self.net(x)

## 2.3 Transformer Block

GPT2's Transformer Block is similar to that of the original transformer architecture:
<p align="left">
    <img src="https://i.postimg.cc/MpQynVfF/add-and-norm.png" alt="drawing" width="800"/>
</p>

GPT2 adds **Layer Normalization** before applying Multi-Head Attention (the Dropout is part of ```MultiHeadAttention``` in this implementation).

In [21]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, ff_dim, dropout)

    def forward(self, x, mask):
        # layer norm, attention, residual
        x = x + self.attn(self.ln1(x), mask)
        # layer norm, feedforward, residual
        x = x + self.ff(self.ln2(x))
        return x

## 2.4 GPT2

After the input is tokenized, tokens are vectorized by summing learnable token embeddings to learnable posistional embeddings:

<p align="left">
    <img src="https://i.postimg.cc/qBGtZQyX/tokens-to-embeddings.png" alt="drawing" width="800"/>
</p>

Then, the vectors are processed by ```num_layers``` transformer blocks, and finally passed to the **Output Layer** which will return the logits for each element of the input sequence (after passing the logits through a softmax, these become the probabilities assigned to each token in the vocabulary).

<p align="left">
    <img src="https://i.postimg.cc/90NcH4Wz/output-layer.png" alt="drawing" width="800"/>
</p>

In [22]:
class GPT2(nn.Module):
    def __init__(
        self,
        vocab_size,
        # GPT2 default hyperparameters
        pos_emb_size=1024,
        d_model=768,
        n_heads=12,
        num_layers=12,
        ff_dim=3072,
        dropout=0.1,
    ):
        super().__init__()
        self.pos_emb_size = pos_emb_size

        # learnable token embeddings
        self.token_embed = nn.Embedding(vocab_size, d_model)
        # learnable positional embeddings
        self.pos_embed = nn.Embedding(pos_emb_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        # output layer
        self.lm_head = nn.Linear(d_model, vocab_size)

        # triangular matrix: (sub)token i will only be able to attend (sub)tokens j, 0 <= j <= i
        mask = torch.tril(torch.ones(pos_emb_size, pos_emb_size)).unsqueeze(0).unsqueeze(0)
        # mask is not a learnable parameter, 
        # but needs to be moved with everything else when doing .to(device) 
        self.register_buffer("mask", mask)
    
    def forward(self, idx):
        B, L = idx.shape # batch size, sequence length
        # positional embedding at most for position self.pos_emb_size
        # longer sequences cannot be processed (the model never learned positional embeddings for positions greater than self.pos_emb_size)
        assert L <= self.pos_emb_size

        pos = torch.arange(L, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(pos)

        mask = self.mask[:, :, :L, :L]

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits  # (B, L, vocab)

# 3 Data loading 

In [23]:
DEVICE = "cuda:0" # it can be changed with 'cpu' if you do not have a gpu

In [24]:
# Loading the corpus 

def read_file(path, eos_token="<eos>"):
    output = []
    with open(path, "r") as f:
        for line in f.readlines():
            output.append(line.strip() + " " + eos_token)
    return output

In [25]:
# If you are using Colab, run these lines
# !wget -P dataset/PennTreeBank https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/dataset/PennTreeBank/ptb.test.txt
# !wget -P dataset/PennTreeBank https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/dataset/PennTreeBank/ptb.valid.txt
# !wget -P dataset/PennTreeBank https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/dataset/PennTreeBank/ptb.train.txt

In [26]:
train_raw = read_file("dataset/PennTreeBank/ptb.train.txt")
dev_raw = read_file("dataset/PennTreeBank/ptb.valid.txt")
test_raw = read_file("dataset/PennTreeBank/ptb.test.txt")

In [27]:
import torch
import torch.utils.data as data

class PennTreeBank (data.Dataset):
    # Mandatory methods are __init__, __len__ and __getitem__
    def __init__(self, corpus):
        self.sents = [sent for sent in corpus]

    def __len__(self):
        return len(self.sents)

    def __getitem__(self, idx):
        return self.sents[idx]

In [28]:
train_dataset = PennTreeBank(train_raw)
dev_dataset = PennTreeBank(dev_raw)
test_dataset = PennTreeBank(test_raw)

In [29]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [30]:
from functools import partial
from torch.utils.data import DataLoader

def collate_fn(batch, tokenizer, device):
    tokenized = tokenizer(batch, padding=True, return_tensors="pt")

    input_ids = tokenized.input_ids[:, :-1].detach().clone().to(device)
    # labels are the input shifted left -> predict the next token
    labels = tokenized.input_ids[:, 1:].detach().clone().to(device)

    # count non-pad tokens
    n_tokens = torch.sum(input_ids != tokenizer.pad_token_id)

    return input_ids, labels, n_tokens

# Dataloader instantiation
# You can reduce the batch_size if the GPU memory is not enough
train_loader = DataLoader(train_dataset, batch_size=8, collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE),  shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE))
test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE))

# 4 Train and validate the model 

In [31]:
import math

from tqdm.notebook import tqdm

def train_loop(data, optimizer, criterion, model):
    model.train()
    loss_array = []
    number_of_tokens = []
    
    pbar = tqdm(data, desc="Training:", unit="batch", total=len(data))

    for i, (input_ids, labels, n_tokens) in enumerate(pbar):
        optimizer.zero_grad() # Zeroing the gradient
        output = model(input_ids)
        # need to reshape as (B, vocab, L)
        loss = criterion(output.permute(0,2,1), labels)
        loss_array.append(loss.item() * n_tokens)
        number_of_tokens.append(n_tokens)
        loss.backward() # Compute the gradient, deleting the computational graph
        optimizer.step() # Update the weights

        if i % 100 == 0:
            pbar.set_postfix(loss=(sum(loss_array)/sum(number_of_tokens)).item())

    return sum(loss_array)/sum(number_of_tokens)

def eval_loop(data, eval_criterion, model):
    model.eval()
    loss_to_return = []
    loss_array = []
    number_of_tokens = []
    # softmax = nn.Softmax(dim=1) # Use Softmax if you need the actual probability
    with torch.no_grad(): # It used to avoid the creation of computational graph
        for input_ids, labels, n_tokens in tqdm(data, desc="Evaluating: ", unit="batch", total=len(data)):
            output = model(input_ids)
            # need to reshape as (B, vocab, L)
            loss = eval_criterion(output.permute(0,2,1), labels)
            loss_array.append(loss.item() * n_tokens)
            number_of_tokens.append(n_tokens)
            
    loss_to_return = sum(loss_array) / sum(number_of_tokens)
    ppl = math.exp(loss_to_return)
    return ppl, loss_to_return

def init_weights(mat):
    for m in mat.modules():
        if type(m) in [nn.Linear]:
            torch.nn.init.uniform_(m.weight, -0.01, 0.01)
            if m.bias != None:
                m.bias.data.fill_(0.01)

In [32]:
import torch.optim as optim

# Don't forget to experiment with a lower training batch size
# Increasing the back propagation steps can be seen as a regularization step

# With AdamW try with a lower learning rate (< 0.1)
lr = 0.1 # This is definitely not good for AdamW

vocab_len = len(tokenizer)

# Experiment also with a smaller or bigger model 
# model = GPT2(
#     vocab_len,
#     pos_emb_size=1024,
#     d_model=20,
#     n_heads=1,
#     num_layers=1,
#     ff_dim=20,
# ).to(DEVICE)
# model.apply(init_weights)

# optimizer = optim.AdamW(model.parameters(), lr=lr)
criterion_train = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
criterion_eval = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [33]:
import matplotlib.pyplot as plt
import copy

# n_epochs = 100
# patience = 3
# losses_train = []
# losses_dev = []
# sampled_epochs = []
# best_ppl = math.inf
# best_model = None
# pbar = tqdm(range(n_epochs))
# #If the PPL is too high try to change the learning rate
# for epoch in pbar:
#     loss = train_loop(train_loader, optimizer, criterion_train, model)    
#     if epoch % 1 == 0:
#         sampled_epochs.append(epoch)
#         losses_train.append(loss.item())
#         ppl_dev, loss_dev = eval_loop(dev_loader, criterion_eval, model)
#         losses_dev.append(loss_dev.item())
#         pbar.set_description("PPL: %f" % ppl_dev)
#         if ppl_dev < best_ppl: # the lower, the better
#             best_ppl = ppl_dev
#             best_model = copy.deepcopy(model).to('cpu')
#             patience = 3
#         else:
#             patience -= 1
            
#         if patience <= 0: # Early stopping with patience
#             break 

# best_model.to(DEVICE)
# final_ppl,  _ = eval_loop(test_loader, criterion_eval, best_model)    
# print('Test ppl: ', final_ppl)

If your model makes you happy and you want to reuse it, you have [to save it and load it](https://pytorch.org/tutorials/beginner/saving_loading_models.html). 
In PyTorch this is straightforward.

In [34]:
## To save the model
#path = 'bin/model_name.pt'
#torch.save(model.state_dict(), path)
## To load the model you need to initialize it
#model = GPT2(vocab_len, pos_emb_size=1024, d_model=20, n_heads=1, num_layers=1, ff_dim=20).to(DEVICE)
## Then you load it
#model.load_state_dict(torch.load(path))
#print(model)


# Mandatory Exam Exercise
## Part 1.A
In this, you have to modify the baseline GPT2 by adding a set of techniques that might improve the performance. In this, you have to __*add one modification at a time incrementally*__. If adding a modification decreases the performance, you can remove it and move forward with the others. However, in the report, you have to provide and comment on this unsuccessful experiment.  For each of your experiments, you have to print the performance expressed with Perplexity (PPL).
<br>
One of the important tasks of training a neural network is  hyperparameter optimization. Thus, you have to play with the hyperparameters to minimise the PPL and thus print the results achieved with the best configuration (in particular <b>the learning rate</b>). 

**Mandatory requirements**: For the following experiments the perplexity must be below 250 (***PPL < 250***).

0. Baseline: find a suitable learning rate, but keep the other hyperparameters fixed.
1. Hyperparameter optimization: incrementally change hyperparameters one at a time (```d_model```, ```n_heads```, ```num_layers```, ```ff_dim```). You may have to readjust the learning rate for larger model sizes.
2. Add dropout layers: --> [link](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html). 
    - one after obtaining the embeddings (after summing token embeddings with positional embeddings)
    - after computing the attention weights
    - after the output projection in ```MultiHeadAttention```
    - after the last linear layer of the Feed Forward
3. Add weight tying: the input embeddings (```token_embed```) and the output layer (```lm_head```) have to share the exact same weights

In [ ]:
# Part 1.A - experiments runner
import math
import copy
from dataclasses import dataclass

# Re-implement a GPT2 variant with optional dropout and weight tying
class MultiHeadAttention_Mod(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, attn_dropout=0.1, proj_dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.h_dim = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj_dropout = nn.Dropout(proj_dropout)

    def forward(self, x, mask):
        B, L, d_model = x.size()

        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        q = q.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)

        similarity = q @ k.transpose(-2, -1)
        similarity = similarity * (1 / torch.sqrt(torch.tensor(self.h_dim, device=x.device)))
        similarity = similarity.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(similarity, dim=-1)
        attn = self.attn_dropout(attn)
        y = attn @ v
        y = y.transpose(1, 2).contiguous().view(B, L, d_model)
        y = self.out_proj(y)
        y = self.proj_dropout(y)
        return y

class FeedForward_Mod(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

class TransformerBlock_Mod(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1, attn_dropout=0.1, proj_dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention_Mod(d_model, n_heads, dropout, attn_dropout, proj_dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward_Mod(d_model, ff_dim, dropout)

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

class GPT2_Mod(nn.Module):
    def __init__(
        self,
        vocab_size,
        pos_emb_size=1024,
        d_model=768,
        n_heads=12,
        num_layers=12,
        ff_dim=3072,
        dropout=0.1,
        attn_dropout=0.1,
        proj_dropout=0.1,
        emb_dropout=0.1,
        weight_tying=False,
    ):
        super().__init__()
        self.pos_emb_size = pos_emb_size
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(pos_emb_size, d_model)
        self.emb_dropout = nn.Dropout(emb_dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock_Mod(d_model, n_heads, ff_dim, dropout, attn_dropout, proj_dropout)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if weight_tying:
            self.lm_head.weight = self.token_embed.weight

        mask = torch.tril(torch.ones(pos_emb_size, pos_emb_size)).unsqueeze(0).unsqueeze(0)
        self.register_buffer("mask", mask)

    def forward(self, idx):
        B, L = idx.shape
        assert L <= self.pos_emb_size
        pos = torch.arange(L, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(pos)
        x = self.emb_dropout(x)
        mask = self.mask[:, :, :L, :L]
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

@dataclass
class ExperimentConfig:
    name: str
    d_model: int = 20
    n_heads: int = 1
    num_layers: int = 1
    ff_dim: int = 20
    lr: float = 0.01
    dropout: float = 0.0
    attn_dropout: float = 0.0
    proj_dropout: float = 0.0
    emb_dropout: float = 0.0
    weight_tying: bool = False
    n_epochs: int = 50
    patience: int = 3
    save_best: bool = False

def run_experiment(cfg: ExperimentConfig):
    print(f"\n=== {cfg.name} ===")
    torch.manual_seed(42)
    model = GPT2_Mod(
        vocab_len,
        pos_emb_size=1024,
        d_model=cfg.d_model,
        n_heads=cfg.n_heads,
        num_layers=cfg.num_layers,
        ff_dim=cfg.ff_dim,
        dropout=cfg.dropout,
        attn_dropout=cfg.attn_dropout,
        proj_dropout=cfg.proj_dropout,
        emb_dropout=cfg.emb_dropout,
        weight_tying=cfg.weight_tying,
    ).to(DEVICE)
    model.apply(init_weights)

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr)
    criterion_train = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    criterion_eval = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    best_ppl = math.inf
    best_model = None
    patience = cfg.patience
    safe_name = cfg.name.replace(" ", "_")
    for epoch in tqdm(range(cfg.n_epochs), desc="Epochs"):
        loss = train_loop(train_loader, optimizer, criterion_train, model)
        ppl_dev, _ = eval_loop(dev_loader, criterion_eval, model)
        if ppl_dev < best_ppl:
            best_ppl = ppl_dev
            best_model = copy.deepcopy(model).to('cpu')
            if cfg.save_best:
                torch.save(model.state_dict(), f"{safe_name}_model.pt")
            patience = cfg.patience
        else:
            patience -= 1
        if patience <= 0:
            break

    best_model.to(DEVICE)
    final_ppl, _ = eval_loop(test_loader, criterion_eval, best_model)
    print(f"Best dev PPL: {best_ppl:.2f} | Test PPL: {final_ppl:.2f}")
    return best_ppl, final_ppl

# === Experiments (edit these as you test) ===
experiments = [
    # 0) Baseline: tune lr only, keep hyperparameters fixed
    ExperimentConfig(name="Baseline", lr=0.01, save_best=True),
    # 1) Hyperparameter optimization (one change at a time)
    ExperimentConfig(name="Bigger d_model", d_model=32, ff_dim=64, lr=0.005, save_best=True),
    ExperimentConfig(name="More heads", n_heads=2, d_model=32, ff_dim=64, lr=0.005, save_best=True),
    ExperimentConfig(name="More layers", num_layers=2, d_model=32, ff_dim=64, lr=0.003, save_best=True),
    # 2) Add dropout layers incrementally
    ExperimentConfig(name="+ emb dropout", d_model=32, ff_dim=64, emb_dropout=0.1, lr=0.003, save_best=True),
    ExperimentConfig(name="+ attn dropout", d_model=32, ff_dim=64, emb_dropout=0.1, attn_dropout=0.1, lr=0.003, save_best=True),
    ExperimentConfig(name="+ proj dropout", d_model=32, ff_dim=64, emb_dropout=0.1, attn_dropout=0.1, proj_dropout=0.1, lr=0.003, save_best=True),
    ExperimentConfig(name="+ ff dropout", d_model=32, ff_dim=64, emb_dropout=0.1, attn_dropout=0.1, proj_dropout=0.1, dropout=0.1, lr=0.003, save_best=True),
    # 3) Add weight tying
    ExperimentConfig(name="+ weight tying", d_model=32, ff_dim=64, emb_dropout=0.1, attn_dropout=0.1, proj_dropout=0.1, dropout=0.1, weight_tying=True, lr=0.003, save_best=True),
]

# strict type validation
if not isinstance(experiments, list) or not all(isinstance(x, ExperimentConfig) for x in experiments):
    raise TypeError(f"experiments must be a list of ExperimentConfig, got {[type(x) for x in experiments]}")

results = []
for cfg in experiments:
    results.append((cfg.name, *run_experiment(cfg)))

print("\nSummary (name, best dev PPL, test PPL):")
for r in results:
    print(r)


=== Baseline ===


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

Training::   0%|          | 0/5259 [00:00<?, ?batch/s]

Evaluating:   0%|          | 0/211 [00:00<?, ?batch/s]

Training::   0%|          | 0/5259 [00:00<?, ?batch/s]

...

Training::   0%|          | 0/5259 [00:00<?, ?batch/s]

Evaluating:   0%|          | 0/211 [00:00<?, ?batch/s]

Training::   0%|          | 0/5259 [00:00<?, ?batch/s]

Evaluating:   0%|          | 0/211 [00:00<?, ?batch/s]

Training::   0%|          | 0/5259 [00:00<?, ?batch/s]

## Part 1.B

While we can train GPT2 from scratch, we may benefit from the pre-trained weights.
However, as you might have noticed from the previous part, training large models requires a large amount of resources, and training the full-sized GPT2 (small) may require more resources than are available to us.

How can we solve this? Instead of fine-tuning all parameters, we can train a small fraction of additional parameters called adapters.
The most popular and widely used technique is Low Rank Adaptatation (LoRA), presented in [this paper](https://arxiv.org/pdf/2106.09685). 

<p align="center">
    <img src="https://i.postimg.cc/tg4B4rV5/Lo-RA.png" alt="drawing" width="400"/>
</p>

Starting from the pre-trained model from Huggingface, fine-tune GPT2 using LoRA (Low Rank Adaptation). ***You have to implement LoRA yourself manually***, libraries such as PEFT are not allowed. Complete the `GPT_LoRA` model we provide below by adding the LoRA adapters and only make these adapters trainable.

- Implement LoRA and fine-tune GPT2 by training only the LoRA adapters. Apply the LoRA adapters to the query, key, and value weight matrices.
- Experiment with the ```rank``` and ```alpha``` hyperparameters.

**Mandatory requirements**: in the experiments with LoRA the perplexity must be below 250 (***PPL < 250***) and it should be lower than that achieved in Part 1.A.

# Models and Training for Part 1.B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# check how many parameters are trainable
def param_stats(model):
    total = sum(param.numel() for param in model.parameters())
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    print(f"total params: {total:,}")
    print(f"trainable params: {trainable:,}")
    print(f"frozen params: {total - trainable:,}")

In [ ]:
from typing import Optional, Tuple, Union
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

class CustomGPT2Attention(GPT2Attention):
    def __init__(self, config, rank, alpha):
        super().__init__(config)
        # Add layers to implement LoRA

    # edit the forward method to implement LoRa
    # from transformers 4.38.0
    # https://github.com/huggingface/transformers/blob/v4.38.0/src/transformers/models/gpt2/modeling_gpt2.py
    def forward(
        self,
        hidden_states: Optional[Tuple[torch.FloatTensor]],
        layer_past: Optional[Tuple[torch.Tensor]] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        encoder_hidden_states: Optional[torch.Tensor] = None,
        encoder_attention_mask: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = False,
        output_attentions: Optional[bool] = False,
    ) -> Tuple[Union[torch.Tensor, Tuple[torch.Tensor]], ...]:
        if encoder_hidden_states is not None:
            if not hasattr(self, "q_attn"):
                raise ValueError(
                    "If class is used as cross attention, the weights `q_attn` have to be defined. "
                    "Please make sure to instantiate class with `GPT2Attention(..., is_cross_attention=True)`."
                )

            query = self.q_attn(hidden_states)
            key, value = self.c_attn(encoder_hidden_states).split(self.split_size, dim=2)
            attention_mask = encoder_attention_mask
        else:
            query, key, value = self.c_attn(hidden_states).split(self.split_size, dim=2)

        query = self._split_heads(query, self.num_heads, self.head_dim)
        key = self._split_heads(key, self.num_heads, self.head_dim)
        value = self._split_heads(value, self.num_heads, self.head_dim)

        if layer_past is not None:
            past_key, past_value = layer_past
            key = torch.cat((past_key, key), dim=-2)
            value = torch.cat((past_value, value), dim=-2)

        if use_cache is True:
            present = (key, value)
        else:
            present = None

        if self.reorder_and_upcast_attn:
            attn_output, attn_weights = self._upcast_and_reordered_attn(query, key, value, attention_mask, head_mask)
        else:
            attn_output, attn_weights = self._attn(query, key, value, attention_mask, head_mask)

        attn_output = self._merge_heads(attn_output, self.num_heads, self.head_dim)
        attn_output = self.c_proj(attn_output)
        attn_output = self.resid_dropout(attn_output)

        outputs = (attn_output, present)
        if output_attentions:
            outputs += (attn_weights,)

        return outputs  # a, present, (attentions)

In [ ]:
from transformers import GPT2LMHeadModel

class GPT2_LoRA(GPT2LMHeadModel):
    def __init__(self, *model_args, rank, alpha, **model_kwargs):
        super().__init__(*model_args, **model_kwargs)
        # add custom attention blocks layers
        for block in self.transformer.h:
            # substitute block.attn with a new instance of CustomGPT2Attention
            # keep the weights from block.attn and apply them to the new instance using .load_state_dict()
            pass
    
    def forward(self, *args, **kwargs):
        return super().forward(*args,**kwargs)

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
# not good parameters for alpha and rank
model = GPT2_LoRA.from_pretrained("openai-community/gpt2", alpha=1, rank=1)
model.to(DEVICE)

## Train and validate

In [ ]:
import math

from tqdm.notebook import tqdm

def train_loop(data, optimizer, model, tokenizer):
    model.train()
    loss_array = []
    number_of_tokens = []
    
    pbar = tqdm(data, desc="Training:", unit="batch", total=len(data))

    for i, (input_ids, _, n_tokens) in enumerate(pbar):
        optimizer.zero_grad() # Zeroing the gradient
        # we don't shift the labels to the left, the model manages it internally
        labels = input_ids.clone().detach()
        # we cannot specify ignore_index, so we replace our pad tokens with -100
        # -100 be ignored by default when the model computes the performance
        labels[labels == tokenizer.pad_token_id] = -100
        output = model(input_ids, labels=labels)
        loss_array.append(output.loss.item() * n_tokens)
        number_of_tokens.append(n_tokens)
        output.loss.backward() # Compute the gradient, deleting the computational graph
        optimizer.step() # Update the weights

        if i % 100 == 0:
            pbar.set_postfix(loss=(sum(loss_array)/sum(number_of_tokens)).item())

    return sum(loss_array)/sum(number_of_tokens)

def eval_loop(data, model, tokenizer):
    model.eval()
    loss_to_return = []
    loss_array = []
    number_of_tokens = []
    with torch.no_grad(): # It used to avoid the creation of computational graph
        for input_ids, _, n_tokens in tqdm(data, desc="Evaluating: ", unit="batch", total=len(data)):
            # we don't shift the labels to the left, the model manages it internally
            labels = input_ids.clone().detach()
            # we cannot specify ignore_index, so we replace our pad tokens with -100
            # -100 be ignored by default when the model computes the performance
            labels[labels == tokenizer.pad_token_id] = -100
            output = model(input_ids, labels=labels)
            loss_array.append(output.loss.item() * n_tokens)
            number_of_tokens.append(n_tokens)
            
    loss_to_return = sum(loss_array) / sum(number_of_tokens)
    ppl = math.exp(loss_to_return)
    return ppl, loss_to_return

In [ ]:
import torch.optim as optim

lr = 0.1 # not good for AdamW

vocab_len = len(tokenizer)

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2_LoRA.from_pretrained("openai-community/gpt2", alpha=1, rank=1) # not good values for LoRA
model.to(DEVICE)

# train only adapter layers
for param in model.parameters():
    # start by freezing all parameters
    #param.requires_grad = False
    pass
for module in model.modules():
    # make only your layers trainable
    #if hasattr(module, "<your_layer_name>"):
    #    for param in module.<your_layer_name>.parameters():
    #        param.requires_grad = True
    pass

# check: print trainable layers
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

optimizer = optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=lr
)

param_stats(model)

In [ ]:
import matplotlib.pyplot as plt
import copy

n_epochs = 100
patience = 3
losses_train = []
losses_dev = []
sampled_epochs = []
best_ppl = math.inf
best_model = None
pbar = tqdm(range(n_epochs))
#If the PPL is too high try to change the learning rate
for epoch in pbar:
    loss = train_loop(train_loader, optimizer, model, tokenizer)    
    if epoch % 1 == 0:
        sampled_epochs.append(epoch)
        losses_train.append(loss.item())
        ppl_dev, loss_dev = eval_loop(dev_loader, model, tokenizer)
        losses_dev.append(loss_dev.item())
        pbar.set_description("PPL: %f" % ppl_dev)
        if ppl_dev < best_ppl: # the lower, the better
            best_ppl = ppl_dev
            best_model = copy.deepcopy(model).to('cpu')
            patience = 3
        else:
            patience -= 1
            
        if patience <= 0: # Early stopping with patience
            break 

best_model.to(DEVICE)
final_ppl,  _ = eval_loop(test_loader, best_model, tokenizer)    
print('Test ppl: ', final_ppl)